# Demo 12d — The recurrent network and its gradient

**Class 12 · R1–R2** (slides: `slides/12_neural_networks.md`)

* **R1:** a vanilla RNN unrolled by hand on three numbers, checked against Keras `SimpleRNN`.
* **R2:** the gradient through time: $\|\partial h_T / \partial h_t\|$ against the lag for four recurrent weights, checked against `jax.jacobian`; gradient clipping on a divergent run.

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")
import jax
import jax.numpy as jnp
import keras
import numpy as np
import polars as pl

from faa import demo, neural, plotlib
from faa.demo import show, slide

demo.setup("12")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 0

## Slide: An RNN by hand (R1)

One input per step ($d = 1$), two hidden units ($u = 2$), $h_t = \tanh(x_t W + h_{t-1} U + b)$, $h_0 = 0$. Input $x = (1, -1, 0.5)$.

In [ ]:
W = np.array([[0.5, -0.4]])
U = np.array([[0.3, 0.2], [-0.1, 0.4]])
b = np.zeros(2)
xs = np.array([[1.0], [-1.0], [0.5]])
h = np.zeros(2)
hs = []
slide("An RNN by hand")
for t, x in enumerate(xs, start=1):
    pre = x @ W + h @ U + b
    h = np.tanh(pre)
    hs.append(h)
    print(
        f"t={t}: x={x[0]:+.1f}  x W = {np.round(x @ W, 3)}  h U = {np.round(hs[-2] @ U if t > 1 else np.zeros(2), 3)}  pre = {np.round(pre, 3)}  h_{t} = {np.round(h, 3)}"
    )
layer = keras.layers.SimpleRNN(2, return_sequences=True)
layer.build((None, 3, 1))
layer.set_weights([W, U, b])
print("Keras SimpleRNN, same weights:", np.round(np.array(layer(xs[None])[0]), 3).tolist())
print("max difference:", float(np.abs(np.array(layer(xs[None])[0]) - np.array(hs)).max()))
print("parameters u(u + d + 1) =", 2 * (2 + 1 + 1))

## Slide: Same weights at every step (R1)

The parameter count does not depend on the length of the sequence; the cost of one pass grows with it.

In [ ]:
rows = []
for kind in ("rnn", "lstm", "gru"):
    for u in (16, 32, 64):
        m = neural.recurrent_model(kind, (50, 5), units=u)
        rows.append({"cell": kind.upper(), "units u": u, "d": 5, "parameters": neural.n_params(m)})
slide("Parameters of a recurrent layer")
show(pl.DataFrame(rows))
f, ax = plotlib.fig("col", 1.6)
for kind, c in zip(("RNN", "LSTM", "GRU"), (RED, BLUE, GREEN)):
    sub = [r for r in rows if r["cell"] == kind]
    ax.plot([r["units u"] for r in sub], [r["parameters"] for r in sub], "o-", color=c, label=kind)
ax.set_xlabel("units $u$ ($d = 5$, with a dense head)")
ax.set_ylabel("parameters")
ax.legend(frameon=False)
plotlib.save(f, "r1_params")
print("sequence length 5 or 500: same parameters, 100x the steps")

## Slide: The gradient through time (R2)

$\partial h_T/\partial h_t = \prod_{k=t+1}^{T} \mathrm{diag}(1 - h_k^2)\, U^\top$. Its norm shrinks or grows geometrically with the lag $T - t$,
at a rate set by the largest singular value of $U$. Four recurrent matrices $U = \rho Q$ ($Q$ orthogonal, so every singular value is $\rho$), $u = 32$.

In [ ]:
u, T = 32, 60


def run(Uw, Ww, x):
    def step(h, xt):
        h = jnp.tanh(xt @ Ww + h @ Uw)
        return h, h

    _, hs = jax.lax.scan(step, jnp.zeros(u), x)
    return hs


rng = np.random.default_rng(SEED)
Q, _ = np.linalg.qr(rng.normal(size=(u, u)))
Ww = jnp.array(rng.normal(0, 0.5, (1, u)))
xin = jnp.array(rng.normal(size=(T, 1)))
curves = {}
for rho in (0.5, 0.9, 1.1, 1.5):
    Uw = jnp.array(rho * Q)
    hs_ = run(Uw, Ww, xin)
    # J_L = M_{T-1} M_{T-2} ... M_{T-L}, with M_k = diag(1 - h_k^2) U^T: the Jacobian of h_T with respect to h_{T-1-L}
    J = np.eye(u)
    norms = []
    for lag in range(1, T):
        J = J @ (np.diag(1 - np.array(hs_[T - lag]) ** 2) @ np.array(Uw).T)
        norms.append(np.linalg.norm(J, 2))
    curves[rho] = np.array(norms)
rho, lag = 0.9, 10
Uw = jnp.array(rho * Q)


def last_state(h0):
    """h_T when the run starts from the state h0 = h_{T-1-lag}, with the same inputs from then on."""

    def step(h, xt):
        h = jnp.tanh(xt @ Ww + h @ Uw)
        return h, h

    return jax.lax.scan(step, h0, xin[T - lag :])[1][-1]


h_t = run(Uw, Ww, xin)[T - 1 - lag]
jac = np.array(jax.jacobian(last_state)(h_t))
exact = np.linalg.norm(jac, 2)
slide("Gradient through time")
print(
    f"rho 0.9, lag 10: spectral norm by jax.jacobian {exact:.6f}, by the product of the one-step Jacobians {curves[0.9][9]:.6f}"
)
rows = [{"lag": lag, **{f"rho={r}": curves[r][lag - 1] for r in curves}} for lag in (1, 5, 10, 20, 40)]
pl.Config.set_fmt_float("mixed")
show(pl.DataFrame(rows), digits=6)
f, ax = plotlib.fig("col", 1.9)
for r, c in zip(curves, (BLUE, GREEN, ORANGE, RED)):
    ax.semilogy(np.arange(1, T), curves[r], color=c, label=f"$\\rho$ = {r}")
ax.set_xlabel("lag $T - t$")
ax.set_ylabel("$\\|\\partial h_T / \\partial h_t\\|$")
ax.legend(frameon=False, fontsize=6)
plotlib.save(f, "r2_gradient_lag")

## Slide: The bound (R2)

$\|\partial h_T/\partial h_t\| \le (\gamma\,\sigma_{\max}(U))^{T-t}$ with $\gamma = \max |1 - h^2| \le 1$ for $\tanh$. The measured curves against the bound $\rho^{\text{lag}}$:

In [ ]:
rows = [{"rho": r, "lag": 20, "measured": curves[r][19], "bound rho^lag": r**20} for r in curves]
slide("The bound")
show(pl.DataFrame(rows), digits=6)
assert all(row["measured"] <= row["bound rho^lag"] * 1.0001 for row in rows)
f, ax = plotlib.fig("col", 1.7)
for r, c in ((0.9, GREEN), (1.1, ORANGE)):
    ax.semilogy(np.arange(1, 41), curves[r][:40], color=c, label=f"measured, $\\rho$ = {r}")
    ax.semilogy(np.arange(1, 41), r ** np.arange(1, 41), color=c, ls="--", label="bound $\\rho^{lag}$")
ax.set_xlabel("lag $T - t$")
ax.set_ylabel("$\\|\\partial h_T / \\partial h_t\\|$")
ax.legend(frameon=False, fontsize=5.5)
plotlib.save(f, "r2_bound")

## Slide: Gradient clipping (R2)

When the gradient explodes, a step with a huge gradient throws the weights away. Clipping rescales the gradient to norm at most $c$: $g \leftarrow g \min(1, c/\|g\|)$.
A vanilla RNN on the memory task of lag 20 with plain SGD and a large learning rate, three seeds each.

In [ ]:
x, yy = neural.memory_task(2000, 20, seed=1)
rows = []
fig, ax = plotlib.fig("col", 1.7)
for clip, col in ((None, RED), (1.0, BLUE)):
    finals = []
    for s in range(3):
        keras.utils.set_random_seed(s)
        m = keras.Sequential(
            [keras.layers.Input(x.shape[1:]), keras.layers.SimpleRNN(32), keras.layers.Dense(4, activation="softmax")]
        )
        m.compile(
            optimizer=keras.optimizers.SGD(0.5, clipnorm=clip),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"],
        )
        hist = m.fit(x, yy, epochs=20, batch_size=64, verbose=0)
        finals.append((hist.history["loss"][-1], hist.history["accuracy"][-1]))
        ax.plot(
            hist.history["loss"],
            color=col,
            lw=0.8,
            label=("no clipping" if clip is None else "clipnorm 1") if s == 0 else None,
        )
    rows.append(
        {
            "clipping": "none" if clip is None else f"clipnorm {clip}",
            "final loss": np.nanmean([a for a, _ in finals]),
            "final accuracy": np.mean([b for _, b in finals]),
            "NaN runs": int(sum(np.isnan(a) for a, _ in finals)),
        }
    )
ax.set_xlabel("epoch")
ax.set_ylabel("training loss")
ax.legend(frameon=False)
plotlib.save(fig, "r2_clipping")
slide("Gradient clipping")
show(pl.DataFrame(rows))